In [ ]:
using DelimitedFiles
using CairoMakie
using GLMakie
using LinearAlgebra
using Random
using CSV
using DataFrames
using Serialization
using GeometryBasics
using QuadGK
using Interpolations

In [2]:
z_boundary_conditions = false  # if z_boundary_conditions = true: the periodic boundary conditions are switched on. 
Randomisation = true

save_h_matrix = false
read_h_matrix = false

read_MF_spectrum_0_1 = true
save_MF_spectrum = false

excitation_on = true # if true, the excitation to v=1 is switched on for some random molecules in the matrix

v_excitation_shift  = 26.8 # cm-1, excitation energy to v=1 state
excitation_fraction = 0.08 #0.08 #0.08 # fraction of molecules that are excited to v=1 state
groundstate_population = 1 - excitation_fraction # fraction of molecules that are in v=0 state

visulize_sim_box = true # if true, the simulation box is visualized

# copy_size::Int64 = 27 # 27
nx::Int64 = 24   # 40#copy_size #24, unit of unit cell length
ny::Int64 = nx     # 40#copy_size #24
nz::Int64 = 15   # round(Int, nx*15/24)     #10#copy_size #15

small_box_size_fraction = 1 # 1.55 # fraction of large box size.
ϵ = 1e-12 # additional distance to the interaction cut-off radius to ensure that the interaction is not cut off
Interaction_sphere_central = false # if true, the interaction sphere is centred at the origin of the small simulation box
Interaction_radius_cutoff  = false # if true, the interaction is cut off at the interaction_cut_off_radius
Interaction_centre_inside_unit_cell = false # false dont use it!!!  if true, the interaction sphere is centred at inside of the unit cell

if Interaction_centre_inside_unit_cell == true
     interaction_cut_off_radius = 1/4*sqrt(3) + ϵ  # interaction radius cut-off in units a0_CO, only one unit cell is considered
else
     interaction_cut_off_radius = sqrt(0.5^2+0.5^2) + ϵ  # interaction radius cut-off in units a0_CO, only one unit cell is considered
end

include("./introduction.jl")
include("./ir_spectra.jl")
include("./ir_spectra_h_matrix.jl")
include("./ir_spectra_centre.jl")
if interaction_cut_off_radius == true
     print("cut-off radius = ", interaction_cut_off_radius * a0_CO, "\n")
end

########################## Read the measured FTIR data ##########################

file_path_p = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_p_pol.txt")
file_p      = open(file_path_p)
header_p    = split(strip(readline(file_p)), '\t')
data_p      = readdlm(file_p, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_p = data_p[:, 1]
A_p = data_p[:, 2]
 
file_path_s = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_s_pol.txt")
file_s      = open(file_path_s)
header_s    = split(strip(readline(file_s)), '\t')
data_s      = readdlm(file_s, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_s = data_s[:, 1]
A_s = data_s[:, 2]

############### Read the measured TAS data of 0-1 and 1-2 transition ###############
file_path_w_scan = joinpath("TAS interleaved spectrum", "TAS_w_scan_final.txt")
file_w_scan      = open(file_path_w_scan)
header_w_scan    = split(strip(readline(file_w_scan)), '\t')
data_w_scan = readdlm(file_w_scan, '\t', Float64, header=false)
v_w_scan = data_w_scan[:, 1]
A_w_scan = data_w_scan[:, 2]

########################## Read simulated mean field data ##########################

if read_MF_spectrum_0_1 == true
     file_path_MF = joinpath("mean field spectrum","MF and data_large_range.txt")
     file_MF      = open(file_path_MF)
     header_MF    = split(strip(readline(file_MF)), '\t')
     data_MF      = readdlm(file_MF, '\t', Float64)   # Read the rest of the file as a matrix of Float64

     νk_MF = data_MF[:, 5]
     ipda_MF = data_MF[:, 6]
     isda_MF = data_MF[:, 8]
end
;

In [3]:
ν0 = 2136.97 # cm-1 #2050.0
νk::Vector{Float64} = collect(ν0- 2*range :step:ν0 + 1*range)
nmols_ml = 4*nx*ny*nz

if Randomisation == true    
    Random.seed!() # seed the random number generator
    num = sign.(rand(nmols_ml) .- 0.5)
    eu_unit_vector = num .* eu # randomised unit vector
else
    eu_unit_vector = eu
end
;

In [4]:
# @time ipda_full, isda_full, ip_full, is_full = ir_spectra(νk, eu_unit_vector, com_ol, Δν)

In [5]:
# fig = Figure(size=(900, 600))

# ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
# ax.xlabelsize, ax.ylabelsize  = 24, 24
# ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
# ax.xticklabelsize, ax.yticklabelsize = 20, 20

# conversion_to_mOD = 1000

# lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
# lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

# Normalisation_exciton = maximum(ipda_full*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
# lines!(ax, νk, ipda_full/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
# lines!(ax, νk, isda_full/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")
# #lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
# #lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

# axislegend(ax, labelsize = 18, position=:lt)
# DataInspector(fig)
# display(fig)

In [6]:
@time ipda_h_matrix, isda_h_matrix, ip_h_matrix, is_h_matrix, h_matrix = ir_spectra_h_matrix(νk, eu_unit_vector, com_ol, Δν)

 count_n2:597179520
34560Matrix{Float64}
(34560, 34560)
2
34560
482.697126 seconds (23.81 G allocations: 1.060 TiB, 6.88% gc time, 0.16% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2135.960397528915 0.8750036072813138 … 0.00015906881390544651 0.00039731537823019594; 0.8750036072813138 2136.960218141841 … 0.00039179170642715435 0.00023247391709163096; … ; 0.00015906881390544651 0.00039179170642715435 … 2136.5417084979763 0.8750036072813138; 0.00039731537823019594 0.00023247391709163096 … 0.8750036072813138 2136.069966967662])

In [7]:
h_matrix

34560×34560 Matrix{Float64}:
 2135.96            0.875004     …     0.000159069     0.000397315
    0.875004     2136.96               0.000391792     0.000232474
    0.875004       -0.875004          -3.36475e-6      0.000290615
   -0.875004        0.875004           0.000237117     9.75778e-5
    0.0            -0.875004          -0.000151654    -0.000379437
    0.109568        0.0          …    -0.000379706    -0.000214838
    0.266093        0.505184           3.88872e-5      0.00032525
    0.505184        0.875004           0.000279074     0.000142587
    0.0             0.109568          -0.000137682    -0.000352562
    0.0330278       0.0               -0.000358215    -0.000190766
    ⋮                            ⋱                  
   -0.000352562    -0.000190766        0.0330278       0.0
    0.000129624    -9.59567e-5        -0.266093       -0.505184
    0.000150184    -5.18497e-5         0.505184        0.875004
    0.000151654     0.000379706        0.0             0.875004

In [8]:
h_matrix_path = joinpath("h-matrix", "h-matrix_" *string(nx)*"x"*string(ny)*"x"*string(nz)*"_"*"z_bound_"*string(z_boundary_conditions)*".bin")
# save h matrix as binary file
if save_h_matrix == true
    # save txt file in folder of force matrix # writedlm(h_matrix_path, h_matrix)
    # Save to a binary file
    open(h_matrix_path, "w") do io
        serialize(io, h_matrix)
        println("Matrix saved in binary format as 'matrix.bin'")
    end
end

In [9]:
# Load from binary file
if read_h_matrix == true
    h_matrix_large = open(h_matrix_path, "r") do io # takes about 1 min after restart
        deserialize(io)
    end
    # println("Matrix loaded: ", h_matrix_large)    
    # get the diagonal elements for the different energies
    h_matrix_large_energies = diag(h_matrix_large)
    fig = Figure(size=(900, 600))
    ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
    lines!(ax, h_matrix_large_energies, color=:red, label = L"energies")
    axislegend(ax, labelsize = 18, position=:lt)
    DataInspector(fig)
    display(fig)
end

In [10]:
# small_box_size_fraction = 1 #1.45 # fraction of large box size

# get the index of each molecules and define the middle position of cuboid 
nx_centre::Int64 = nx÷(small_box_size_fraction)   # get simulation subset of alpha-CO which is smaller than small_box_size_fraction-times the size of the total simulation box (use modulo ÷ to get unit cell below exactly small_box_size_fraction the size)
ny_centre::Int64 = ny÷(small_box_size_fraction) 
nz_centre::Int64 = nz

interaction_cut_off_radius = nz_centre / 2

println(nx_centre,' ', ny_centre,' ', nz_centre, " interaction_cut_off_radius: ", interaction_cut_off_radius)
nmols_centre = 4*nx_centre*ny_centre*nz_centre

24 24 15 interaction_cut_off_radius: 7.5


34560

In [11]:
max_x_centre = maximum(com_ol[:, 1]) * nx_centre / nx
max_y_centre = maximum(com_ol[:, 2]) * ny_centre / ny
max_z_centre = maximum(com_ol[:, 3]) * nz_centre / nz
print(max_x_centre, ' ', max_y_centre, ' ', max_z_centre)

23.5 23.5 14.5

In [12]:
# indexing of molecules in the centre of the simulation box:
com_ol_centre         = zeros(Float64, nmols_centre, 3) 
eu_unit_vector_centre = Vector{Vector{Float64}}(undef, nmols_centre)
large_h_matrix_sum    = zeros(Float64, nmols_centre)
position_z_centre = []
index_rows = []

count = 1
for row_i in 1:size(com_ol, 1)
    if com_ol[row_i, 1] <= max_x_centre && com_ol[row_i, 2] <= max_y_centre && com_ol[row_i, 3] <= max_z_centre
        com_ol_centre[count, 1] = com_ol[row_i, 1]
        com_ol_centre[count, 2] = com_ol[row_i, 2]
        com_ol_centre[count, 3] = com_ol[row_i, 3]
        eu_unit_vector_centre[count] = eu_unit_vector[row_i]
        large_h_matrix_sum[count]    = h_matrix[row_i, row_i]
        
        push!(index_rows, row_i) # get the index of the molecules in the centre of the simulation box

        count += 1
        # below get interaction centre
        if nz % 2 == 0  # z even
            if Interaction_sphere_central == false && com_ol[row_i, 1] == 0.0 && com_ol[row_i, 2] == 0.0 && com_ol[row_i, 3] == (max_z_centre*2+1)/4 # get most central molecule in z-direction, x=0, y=0
                position_z_centre = com_ol[row_i, :]
            elseif Interaction_sphere_central == true && com_ol[row_i, 1] == nx_centre / 2 && com_ol[row_i, 2] == ny_centre / 2 && com_ol[row_i, 3] == (max_z_centre*2+1)/4 # get most central molecule in all directions
                position_z_centre = com_ol[row_i, :]
            end
        else # z odd
            if Interaction_sphere_central == false && com_ol[row_i, 1] == 0.0 && com_ol[row_i, 2] == 0.0 && com_ol[row_i, 3] == (max_z_centre*2-1)/4 # get most central molecule in z-direction, x=0, y=0
                position_z_centre = com_ol[row_i, :]
            elseif Interaction_sphere_central == true && com_ol[row_i, 1] == nx_centre / 2 && com_ol[row_i, 2] == ny_centre / 2 && com_ol[row_i, 3] == (max_z_centre*2-1)/4 # get most central molecule in all directions
                position_z_centre = com_ol[row_i, :]
            end
        end
    end
end

if Interaction_centre_inside_unit_cell == true
    position_z_centre_unshifted = position_z_centre * a0_CO
    position_z_centre = position_z_centre * a0_CO .+ a0_CO/4  # m, unit conversion, +a0_CO/4 to get the centre of the interaction radius without cutting off unit-cell
else
    position_z_centre = position_z_centre * a0_CO  # m, unit conversion
end

3-element Vector{Float64}:
 0.0
 0.0
 3.948e-9

In [13]:
# get v=1 molecules randomised 
if excitation_on == true
    num_excited = round(Int, excitation_fraction * nmols_centre)
    excited_indices = randperm(nmols_centre)[1:num_excited]
 end;

In [14]:

if visulize_sim_box == true  # visualize the simulation box:

    X = com_ol[:, 1]
    Y = com_ol[:, 2]
    Z = com_ol[:, 3]

    U = [vec[1] for vec in eu_unit_vector]
    V = [vec[2] for vec in eu_unit_vector]
    W = [vec[3] for vec in eu_unit_vector]

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1]
    Y_centre = com_ol_centre[:, 2]
    Z_centre = com_ol_centre[:, 3]

    U_centre = [vec[1] for vec in eu_unit_vector_centre]
    V_centre = [vec[2] for vec in eu_unit_vector_centre]
    W_centre = [vec[3] for vec in eu_unit_vector_centre]

    # Define sphere properties
    center = Point3f0(position_z_centre[1] / a0_CO, position_z_centre[2]/ a0_CO, position_z_centre[3]/ a0_CO)  # Center of the sphere
    r_0::Float32 = interaction_cut_off_radius  # Radius of the sphere
    sphere = HyperSphere(center, r_0)  # Define a sphere
    
    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate", ylabel = "Y Coordinate", zlabel = "Z Coordinate")

    arrows!(ax,  vec(X), vec(Y), vec(Z), 0.2 .* vec(U), 0.2 .* vec(V), 0.2 .* vec(W), arrowsize=0.1, color=:red)
    scatter!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), markersize=40)
    arrows!(ax,  vec(X_centre), vec(Y_centre), vec(Z_centre), 0.2 .* vec(U_centre), 0.2 .* vec(V_centre), 0.2 .* vec(W_centre), arrowsize=0.15, color=:blue)
    mesh!(ax, sphere, color = (:orange, 0.8))  # Blue color with 50% transparency

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [15]:
if visulize_sim_box == true# && Interaction_radius_cutoff == true
    # visualize the simulation box:

    X = com_ol[:, 1] .* a0_CO
    Y = com_ol[:, 2] .* a0_CO
    Z = com_ol[:, 3] .* a0_CO

    U = [vec[1] for vec in eu_unit_vector] .* a0_CO
    V = [vec[2] for vec in eu_unit_vector] .* a0_CO
    W = [vec[3] for vec in eu_unit_vector] .* a0_CO

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1] .* a0_CO
    Y_centre = com_ol_centre[:, 2] .* a0_CO
    Z_centre = com_ol_centre[:, 3] .* a0_CO

    U_centre = [vec[1] for vec in eu_unit_vector_centre] .* a0_CO
    V_centre = [vec[2] for vec in eu_unit_vector_centre] .* a0_CO
    W_centre = [vec[3] for vec in eu_unit_vector_centre] .* a0_CO

    # Define sphere properties
    center = Point3f0(position_z_centre[1]*1e10, position_z_centre[2]*1e10, position_z_centre[3]*1e10)  # Center of the sphere
    r_0::Float32 = interaction_cut_off_radius * a0_CO *1e10  # Radius of the sphere
    sphere = HyperSphere(center, r_0)  # Define a sphere

    # molecules with excited_indices inside the small box 
    X_excited = com_ol_centre[excited_indices, 1] .* a0_CO
    Y_excited = com_ol_centre[excited_indices, 2] .* a0_CO
    Z_excited = com_ol_centre[excited_indices, 3] .* a0_CO

    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate / Å", ylabel = "Y Coordinate / Å", zlabel = "Z Coordinate / Å")#, limits = ((0, 100), (0, 100), (0, 100)))

    arrows!(ax,  vec(X) * 1e10, vec(Y) * 1e10, vec(Z) * 1e10, 0.2 .* vec(U) * 1e10, 0.2 .* vec(V) * 1e10, 0.2 .* vec(W) * 1e10, arrowsize=0.1, color=:red)
    scatter!(ax, vec(X_centre) * 1e10, vec(Y_centre) * 1e10, vec(Z_centre) * 1e10, markersize=20)
    if excitation_on == true
        scatter!(ax, vec(X_excited) * 1e10, vec(Y_excited) * 1e10, vec(Z_excited) * 1e10, markersize=20)#, color=:Base.get_preferences)
    end
    arrows!(ax, vec(X_centre) * 1e10, vec(Y_centre) * 1e10, vec(Z_centre) * 1e10, 0.2 .* vec(U_centre) * 1e10, 0.2 .* vec(V_centre) * 1e10, 0.2 .* vec(W_centre) * 1e10, arrowsize=0.15, color=:blue)

    #mesh!(ax, sphere, color = (:orange, 0.8))  # Blue color with 50% transparency

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [16]:
sigma_01 = 1.0 # absorption cross section of the 0-1 transition

1.0

In [17]:
# read the force matrix for the maximum size and do eigen(h) operation of small simulation box with entries of the large h matrix on the diagonal
@time ipda, isda, ip, is, eigenvecs, h_centre = ir_spectra_centre(νk, eu_unit_vector_centre,  com_ol_centre, Δν, nmols_centre, large_h_matrix_sum)

2342.970745 seconds (21.28 G allocations: 1.031 TiB, 1.93% gc time, 0.04% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [6.325849946254204e-8 8.295315986965287e-8 … -0.0032718607888144397 0.0010121118142398873; -3.0682520345041796e-6 6.875817165817805e-7 … -0.0038568735274996397 0.0012734452679489757; … ; -2.2708617555444516e-7 -8.756377489075204e-6 … -0.001467474552334128 0.0012493854373745502; -5.423128054622684e-7 -8.661254278347038e-6 … -0.001857195718492052 0.0010825624192844115], [2135.960397528915 0.8750036072813138 … 0.00015906881390544651 0.00039731537823019594; 0.8750036072813138 2136.960218141841 … 0.00039179170642715435 0.00023247391709163096; … ; 0.00015906881390544651 0.000391

In [18]:
α = 0*degrees 

ipda_α = (cos(α))^2 .* ipda + (sin(α))^2 .* isda 
isda_α = (cos(α))^2 .* isda + (sin(α))^2 .* ipda;

In [72]:
# CairoMakie.activate!()
# GLMakie.activate!()
fig = Figure(size=(900, 600))

ax = Axis(fig[1,1], xlabel = L"Wavenumber/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

conversion_to_mOD = 1000

lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
wavenumber_shift = 0# 1.5 # cm-2

lines!(ax, νk.+wavenumber_shift, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
lines!(ax, νk.+wavenumber_shift, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")

#lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
#lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig);

In [77]:
# fold the simulated spectrum ipda with gaussian function with gssn(ν, ν0, Δν) with Δν = 2.3 cm-1 instead of 0.2 cm-1 from FTIR data:
Δν = 2.3e-1 #3*2.3*1e-9 # cm-1

function gaussian(x, μ, σ)
     return @. exp(-0.5 * ((x - μ) / σ)^2) / (σ * sqrt(2π))
end

function fold_spectrum(spectrum, x_values, width)
     folded_spectrum = zeros(length(spectrum))
     for i in 1:length(spectrum)
          folded_spectrum .+= spectrum[i] .* gaussian(x_values, x_values[i], width)
     end
     return folded_spectrum
end

ipda_folded = fold_spectrum(ipda, νk, Δν)    #post
ipda_MF_folded = fold_spectrum(ipda_MF, νk, Δν); #pre  

In [82]:
# difference signal:

fig = Figure(size=(900, 600))
ax = Axis(fig[1,1], xlabel = L"Wavenumber/cm$^{-1}$", ylabel = L"ΔA/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

#lines!(ax, νk.+wavenumber_shift, ipda, label = L"p-pol modelled post$ $")
#lines!(ax, νk.+wavenumber_shift, ipda_folded, label = L"p-pol modelled post folded$ $")
#lines!(ax, νk.+wavenumber_shift, isda, label = L"s-pol modelled post$ $")

#lines!(ax, νk.+wavenumber_shift, ipda_MF, label = L"p-pol modelled pre$ $")
#lines!(ax, νk.+wavenumber_shift, ipda_MF_folded, label = L"p-pol modelled pre folded$ $")
#lines!(ax, νk.+wavenumber_shift, isda_MF, label = L"s-pol modelled pre$ $")

scaling = 2e-1 # scaling for difference spectrum to match the TAS measured data
scaling_FTIR_spectrum = 0.8

#lines!(ax, νk.+wavenumber_shift, (ipda - ipda_MF)*scaling, label = L"p-pol modelled dif$ $")
lines!(ax, νk.+wavenumber_shift, (ipda_folded*1 - ipda_MF_folded*scaling_FTIR_spectrum)*scaling, label = L"p-pol modelled dif folded$ $")

lines!(ax, v_w_scan, A_w_scan, label = L"p-pol modelled dif measured$ $")

#lines!(ax, νk.+wavenumber_shift, isda*scaling - isda_MF, label = L"s-pol modelled dif$ $")

#lines!(ax, νk_MF.+wavenumber_shift, ipda_MF/maximum(ipda_MF)*maximum(ipda), label = L"p-pol modelled pre$ $")
#lines!(ax, νk_MF.+wavenumber_shift, isda_MF/maximum(ipda_MF)*maximum(ipda), label = L"s-pol modelled pre$ $")


axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig);

In [ ]:
# Integral check:
x_min = 2104 # 2104 # 2125 # 
x_max = 2125 # 2115 # 2155 #

x_min = 2125 # 2104 # 2125 # 
x_max = 2155 # 2115 # 2155 #

dif = (ipda_folded*1 - ipda_MF_folded*scaling_FTIR_spectrum)*scaling
itp = LinearInterpolation(νk, dif, extrapolation_bc=Flat())
integral, error = quadgk(x -> itp(x), x_min, x_max)
println("Integral of the difference signal: ", integral)

# Integrate A_w_scan as a function of v_w_scan
itp_w_scan = LinearInterpolation(v_w_scan, A_w_scan, extrapolation_bc=Flat())
integral_w_scan, error_w_scan = quadgk(x -> itp_w_scan(x), x_min, x_max)
println("Integral of the TAS data: ", integral_w_scan)

# Integrate the difference signal over the range of the TAS data
fig = Figure(size=(900, 600))
ax = Axis(fig[1,1], xlabel = L"Wavenumber/cm$^{-1}$", ylabel = L"ΔA/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

lines!(ax, x, y, label = L"Difference signal$ $")

# Highlight the integrated area for the difference signal
x_subset = νk[(νk .>= x_min) .& (νk .<= x_max)]
y_subset = dif[(νk .>= x_min) .& (νk .<= x_max)]
fill_between!(ax, x_subset, 0, y_subset, color = (:blue, 0.3))

# Plot the integral_w_scan curve
lines!(ax, v_w_scan, A_w_scan, color=:red, label = L"TAS data$ $")

# Highlight the area under the TAS data curve in the range x_min to x_max
v_w_scan_subset = v_w_scan[(v_w_scan .>= x_min) .& (v_w_scan .<= x_max)]
A_w_scan_subset = A_w_scan[(v_w_scan .>= x_min) .& (v_w_scan .<= x_max)]
fill_between!(ax, v_w_scan_subset, 0, A_w_scan_subset, color = (:red, 0.3))

axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig);

In [137]:
#integral_w_scan

integral

5.186006782316749

In [138]:
if save_MF_spectrum == true
     # Find the maximum length
     max_length = maximum(length.([v_p, A_p, v_s, A_s, νk, ipda, νk, isda]))

     pad_vector(v, len) = vcat(v, fill(NaN, len - length(v)))

     data_matrix = hcat(pad_vector(v_p, max_length), 
                        pad_vector(A_p, max_length), 
                        pad_vector(v_s, max_length), 
                        pad_vector(A_s, max_length), 
                        pad_vector(νk,   max_length), 
                        pad_vector(ipda, max_length), 
                        pad_vector(νk,   max_length), 
                        pad_vector(isda, max_length))

     header = "v_p\tA_p\tv_s\tA_s\tνk\tipda\tνk\tisda"

     mkpath("mean field spectrum") # Ensure the subfolder exists

     open(joinpath("mean field spectrum","MF and data_2.txt"), "w") do io
          write(io, header * "\n")
          writedlm(io, data_matrix, '\t')
     end
end